# Table III

This notebook generates table III of the paper from the provided assignments.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
"""
Compute the number of intra- and inter-node data movements for one configuration.
"""
def compute_intra_inter(original_csv: Path, assignment_csv: Path, cores: int, hosts: int) -> tuple[int, int]:
    original = pd.read_csv(original_csv, index_col=0).iloc[:, 0].to_numpy(dtype=int)
    assignment = pd.read_csv(assignment_csv, index_col=0).to_numpy(dtype=int)

    if assignment.shape[0] != original.shape[0]:
        raise ValueError(f'Sample count mismatch: {original_csv} vs {assignment_csv}')

    processors_per_host = cores // hosts
    if processors_per_host * hosts != cores:
        raise ValueError(f'Invalid grouping: cores={cores}, hosts={hosts}')

    # Processor i belongs to node group floor(i / processors_per_host).
    group_of = np.arange(cores) // processors_per_host

    source = original[:, None]
    moved = source != assignment
    same_group = group_of[source] == group_of[assignment]

    # Multiply by 2 to account for source + target.
    intra = int((moved & same_group).sum() * 2)
    inter = int((moved & ~same_group).sum() * 2)
    return intra, inter

In [9]:
# (cores, resolution, hosts)
configurations = [
    (144, 48, 4),
    (144, 90, 4),
    (144, 180, 4),
    (576, 48, 16),
    (576, 90, 16),
    (576, 180, 16),
]

rows = []

In [10]:
for cores, resolution, hosts in configurations:
    original_csv = f'original/c{resolution}_p{cores}.csv'
    assignment_csv = f'global_greedy/c{resolution}_p{cores}/assignment.csv'

    intra, inter = compute_intra_inter(
        original_csv=original_csv,
        assignment_csv=assignment_csv,
        cores=cores,
        hosts=hosts,
    )

    total = intra + inter
    rows.append({
        'Cores': cores,
        'Resolution': f'C{resolution}',
        'Intra': intra,
        'Inter': inter,
        '% Inter': 100.0 * inter / total if total else 0.0,
    })

In [11]:
summary = pd.DataFrame(rows)
summary['Intra'] = summary['Intra'].map(lambda v: f'{v:,}')
summary['Inter'] = summary['Inter'].map(lambda v: f'{v:,}')
summary['% Inter'] = summary['% Inter'].map(lambda v: f'{v:.2f}%')

display(summary)

,Cores,Resolution,Intra,Inter,% Inter
0,144,C48,"528,588","1,866,616",77.93%
1,144,C90,"1,803,848","6,489,760",78.25%
2,144,C180,"7,087,804","25,682,664",78.37%
3,576,C48,"114,116","2,926,436",96.25%
4,576,C90,"50,756","1,405,504",96.51%
5,576,C180,"1,414,892","37,446,560",96.36%
